In [1]:
import math
import sys
import yaml
import numpy as np
sys.path.append('../../python/')  
from periphery import logicGate
from periphery import constant
from periphery.Technology import Technology
from periphery.sramWriteDriver import SRAMWriteDriver
from periphery.precharger import Precharger
from periphery.WLdecoder import RowDecoder
from periphery.SenseAmp import SenseAmp
from periphery.DFF import DFF
from periphery.MUX import Mux
from periphery.levelShifter import LevelShifter
from periphery.WLDecoderDriver import WLNewDecoderDriver
from periphery.adder import Adder
from periphery.ADC import SarADC
from periphery.RNGBlock import RNG_block
from periphery.Bus import Bus
from periphery.HTree import HTree
from simulator.subarray import SubArray
from simulator.MAT import MAT
from simulator.Bank import Bank

print(constant.INV)

0
0


In [2]:
with open('../../config.yaml', 'r') as file:
    config = yaml.safe_load(file)

with open('../../mapping.yaml', 'r') as file:
    mapping = yaml.safe_load(file)

with open('../../param.yaml', 'r') as file:
    param = yaml.safe_load(file)

with open('../../RNG.yaml', 'r') as file:
    RNG = yaml.safe_load(file)

In [15]:
class PM:
    def __init__(self, tech,config, param, mapping, RNG,numBankRow,numBankCol, numMATRow, numMATCol, numSubArrayRow, numSubArrayCol, numCol, numRow, num_mu, num_sigma):

        self.tech = tech
        self.param = param
        self.config = config
        self.mapping = mapping
        self.RNG = RNG
        self.initialized = False
        self.numBankRow = numBankRow
        self.numBankCol = numBankCol
        self.numMATRow = numMATRow
        self.numMATCol = numMATCol
        self.num_col = numCol
        self.num_row = numRow
        self.numSubArrayRow = numSubArrayRow
        self.numSubArrayCol = numSubArrayCol
        self.num_mu = num_mu
        self.num_sigma = num_sigma

        self.feature_size = self.tech.get_param('featureSize')
        self.pnSizeRatio = self.tech.get_param('pnSizeRatio')
        self.temp = self.config['temperature']
        self.vdd = self.tech.get_param('vdd')
        self.cell_type = self.config['device_type']
        self.clk_freq = self.config['frequency']

        self.unitWireRes = self.param['unitLengthWireResistance']
        self.wireWidth = self.param['wireWidth']
        self.unitWireCap = 0.2e-15 / 1e-6  # 0.2 fF/um = 0.2e-15 F/micron

        self.resistanceOn = self.param['resistanceOn']
        self.resistanceOff = self.param['resistanceOff']
        self.resistanceAvg = (self.resistanceOn + self.resistanceOff) / 2
        self.writeVoltage = self.param['writeVoltage']
        self.readVoltage = self.param['readVoltage']
        self.accessVoltage = self.param['accessVoltage']
        self.avgWeightBit = self.param['cellBit']
        self.accesstype = self.param['accesstype']
        self.mem_mode = self.param['operationmode']
        self.readPulseWidth = self.param['readPulseWidth']
        self.heightInFeatureSizeSRAM = self.param['heightInFeatureSizeSRAM']
        self.widthInFeatureSizeSRAM = self.param['widthInFeatureSizeSRAM']
        self.widthSRAMCellNMOS = self.param['widthSRAMCellNMOS']
        self.widthSRAMCellPMOS = self.param['widthSRAMCellPMOS']
        self.widthAccessCMOS = self.param['widthAccessCMOS']
        self.minSenseVoltage = self.param['minSenseVoltage']
        self.heightInFeatureSize1T1R = self.param['heightInFeatureSize1T1R']
        self.heightInFeatureSizeCrossbar = self.param['heightInFeatureSizeCrossbar']
        self.widthInFeatureSize1T1R = self.param['widthInFeatureSize1T1R']
        self.widthInFeatureSizeCrossbar = self.param['widthInFeatureSizeCrossbar']

        if self.cell_type == 'SRAM':
            self.heightInFeatureSize = self.heightInFeatureSizeSRAM
            self.widthInFeatureSize = self.widthInFeatureSizeSRAM
        else:
            self.heightInFeatureSize = self.heightInFeatureSize1T1R if self.accessType == 'CMOS_access' else self.heightInFeatureSizeCrossbar

        self.Bank = Bank(numMATRow=self.numMATRow,numMATCol=self.numMATCol,numSubArrayRow=self.numSubArrayRow,numSubArrayCol=self.numSubArrayCol,numCol=self.num_col,numRow=self.num_row,num_mu=self.num_mu,num_sigma=self.num_sigma,param=self.param,mapping=self.mapping,config=self.config,RNG=self.RNG,tech=self.tech)
        self.HTree = HTree(num_row=self.numBankRow,num_col=self.numBankCol,delay_tolerance=0,bus_width=self.num_col,param=self.param,config=self.config,tech=self.tech)
        

        self.initialized = True

    def calculate_area(self, overlap=False):
        if not self.initialized:
            raise ValueError("Bus must be initialized before calculating area.")
        self.Bank_area, self.Bank_height, self.Bank_width = self.Bank.calculate_area(overlap=False)
        
        self.HTree_area = self.HTree.calculate_area(unit_height=self.Bank_height , unit_width = self.Bank_width, folded_ratio=1)

        area = self.numBankRow * self.numBankCol * self.Bank_area + self.HTree_area

        height = math.sqrt(area)
        width = area / height

        return area, height, width

    def calculate_latency(self, num_read):
        if not self.initialized:
            raise ValueError("Bus must be initialized before calculating latency.")
        Bank_read_latency = self.Bank.calculate_latency(num_read = 1)
        HTree_read_latency = self.HTree.calculate_latency(x_init=0,y_init=0,x_end=0,y_end=0,unit_height=self.Bank_height,unit_width=self.Bank_width,num_read=1)
        
        read_latency = Bank_read_latency + HTree_read_latency
        return read_latency * num_read

    def calculate_power(self,input_vector,weight_matrix, num_bit_access, num_read):
        if not self.initialized:
            raise ValueError("Bus must be initialized before calculating power.")
        Bank_read_energy,Bank_leakage = self.Bank.calculate_power(input_vector,weight_matrix, num_bit_access = 32, num_read = 1)
        HTree_read_energy,HTree_leakage = self.HTree.calculate_power(x_init=0,y_init=0,x_end=0,y_end=0,unit_height=self.Bank_height,unit_width=self.Bank_width, num_Bit_Access=1, num_read=1)
        
        read_dynamic_energy = Bank_read_energy + HTree_read_energy
        leakage = Bank_leakage + HTree_leakage
        

        return read_dynamic_energy * num_bit_access * num_read, leakage


In [16]:
tech45 = Technology(node_nm=45, roadmap='HP')
# Instantiate and initialize Precharger
pre = PM(
    numBankRow=2,
    numBankCol=2,
    numMATRow=2,
    numMATCol=2,
    numSubArrayRow=2,
    numSubArrayCol=2,
    numCol=32,
    numRow=128,
    num_mu=16,
    num_sigma=16,
    param=param,
    mapping=mapping,
    config=config,
    RNG=RNG,
    tech=tech45
)

width_array 4.032000000000001e-05
height_array 5.7600000000000004e-05
array_area 2.3224320000000004e-09
pre_charge_area 4.136832000000001e-11
sram_write_driver_area 5.515776000000001e-11
WL_decoder_area 8.8252416e-10
SarADC_area 1.9616469398969537e-09
dff_area 2.7578880000000007e-10
sense_amp_area 4.136832000000001e-11
RNG_bloclk_area 3.634386756923078e-10


In [17]:
pre_charge_area,height,width = pre.calculate_area(overlap=False)

print("Area Result:", pre_charge_area)

Area Result: 9.062540106284899e-07


In [18]:
read_latency = pre.calculate_latency(num_read = 1)
print("Read Latency:", read_latency)

cell_type SRAM
mem_mode conventionalSequential
wl_decoder_read_latency 2.42488307884878e-10
precharger_read_latency 9.866218898852135e-11
col_delay 7.01758472378664e-11
SarADC_read_latency 4.906890595608518e-09
Read Latency: 1.405670611327864e-08


In [19]:
#######################################################################
weight_matrix = np.load('../../slice.npy')
# weight_matrix = np.load('../../conductance.npy')
# weight_matrix = 1 / weight_matrix
num_rows = weight_matrix.shape[0]
input_vector = [0] * num_rows
input_vector[4] = 1   
print("Input Vector Length:", len(input_vector))
#######################################################################
read_energy,leakage = pre.calculate_power(input_vector,weight_matrix, num_bit_access = 32, num_read = 1)
print(f"  Read Dynamic Energy: {read_energy:.3e} J")
print(f"  Leakage Power: {leakage:.3e} W")

Input Vector Length: 64
shape of columnResistance 64
  Read Dynamic Energy: 3.800e-06 J
  Leakage Power: 1.074e-04 W
